# Dataset experiment — train on SCATTERED edits

Only the TRAINING DATA changes vs baseline (append-only Wikipedia). Same architecture,
same losses, same hyperparameters. PURE — no novelty labels in the objective, no oracle.
Tests whether append-only data was what stopped delta from learning "what B adds beyond A."

Source: IteraTeR_full_sent (sentence-level scattered edits; ParaRev fallback). The exact
IteraTeR_human_sent EVAL pairs are EXCLUDED from training (leakage guard).

Success = the **specialization gap** (novel vs shared full-effect; baseline = -0.14)
flips toward/positive when evaluated on IteraTeR_human_sent (held-out).

Saves to `wiki_model_wae.pt`; baseline `wiki_model.pt` untouched. GPU ON, Internet ON.

In [ ]:
# Cell 1 - install + clone/pull repo
!pip install transformers scikit-learn scipy datasets -q

import os, sys
from pathlib import Path

REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')

sys.path.insert(0, str(REPO / 'delta_system'))
print('Files:', sorted([f.name for f in (REPO/'delta_system').glob('*.py')]))

In [ ]:
# Cell 2 - train BASELINE G on scattered edits (~30-60 min on T4). saves wiki_model_wae.pt.
# Only the data differs from baseline. Watch for 'using <dataset> cols=(...)' and
# 'Loaded N scattered pairs ... eval-overlap skipped'.
import subprocess, sys

cmd = [
    sys.executable,
    '/kaggle/working/multihop-retrieval/delta_system/train_wae.py',
    '--steps', '2000',
]
r = subprocess.run(cmd)
print('exit code', r.returncode)

In [ ]:
# Cell 3 - eval on IteraTeR_human_sent (held-out). NO --vib (baseline arch).
# Look at [1] novel vs shared (specialization gap, baseline -0.14) and [4] A-dependence.
import subprocess, sys

cmd = [
    sys.executable,
    '/kaggle/working/multihop-retrieval/delta_system/insertion_cloze_eval.py',
    '--ckpt', '/kaggle/working/checkpoints/wiki_model_wae.pt',
    '--n',    '500',
]
r = subprocess.run(cmd)
print('exit code', r.returncode)